# 03 — EDA & Baseline Model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

In [ ]:
spark = SparkSession.builder.appName("hit-predictor").getOrCreate()
df = pd.read_csv("../data/processed/features.csv")
sdf = spark.createDataFrame(df)

FEATURE_COLS = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo"
]

In [ ]:
df["label"].value_counts().plot(kind="bar", color=["steelblue", "tomato"])
plt.xticks([0, 1], ["Non-Hit", "Hit"], rotation=0)
plt.title("Class Distribution")
plt.ylabel("Count")
plt.show()

In [ ]:
for feat in FEATURE_COLS:
    sns.kdeplot(data=df, x=feat, hue="label", common_norm=False)
    plt.title(feat)
    plt.show()

In [ ]:
sns.heatmap(df[FEATURE_COLS].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlations")
plt.show()

In [ ]:
assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features")
train, test = sdf.randomSplit([0.8, 0.2], seed=42)

In [ ]:
hit = train.filter("label = 1").count()
total = train.count()
ratio = (total - hit) / hit

train = train.withColumn("weight", when(col("label") == 1, ratio).otherwise(1.0))
lr = LogisticRegression(labelCol="label", featuresCol="features", weightCol="weight")

pipeline = Pipeline(stages=[assembler, scaler, lr])
model = pipeline.fit(train)

In [ ]:
predictions = model.transform(test)
auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(predictions)
f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1").evaluate(predictions)
print(f"AUC: {auc:.3f} | F1: {f1:.3f}")